<a href="https://colab.research.google.com/github/KeerthanaSistla/BigDataAnalytics/blob/main/BDA_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark

In [20]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("Advanced Hive Simulation").getOrCreate()

In [21]:
data = [
    (1, "Keerthana", "Math", 85, "Sem1", "Internal"),
    (1, "Keerthana", "Physics", 78, "Sem1", "External"),
    (2, "Rahul", "Math", 45, "Sem1", "Internal"),
    (2, "Rahul", "Physics", 55, "Sem1", "External"),
    (3, "Anjali", "Math", 92, "Sem2", "Internal"),
    (3, "Anjali", "Physics", 88, "Sem2", "External"),
    (4, "Arjun", "Math", 30, "Sem2", "Internal"),
    (4, "Arjun", "Physics", 40, "Sem2", "External")
]

columns = ["student_id", "name", "subject", "marks", "semester", "exam_type"]

df = spark.createDataFrame(data, columns)
df.show()

+----------+---------+-------+-----+--------+---------+
|student_id|     name|subject|marks|semester|exam_type|
+----------+---------+-------+-----+--------+---------+
|         1|Keerthana|   Math|   85|    Sem1| Internal|
|         1|Keerthana|Physics|   78|    Sem1| External|
|         2|    Rahul|   Math|   45|    Sem1| Internal|
|         2|    Rahul|Physics|   55|    Sem1| External|
|         3|   Anjali|   Math|   92|    Sem2| Internal|
|         3|   Anjali|Physics|   88|    Sem2| External|
|         4|    Arjun|   Math|   30|    Sem2| Internal|
|         4|    Arjun|Physics|   40|    Sem2| External|
+----------+---------+-------+-----+--------+---------+



In [22]:
df.write.partitionBy("semester", "exam_type") \
    .mode("overwrite") \
    .csv("advanced_partitioned_data")

In [23]:
df = df.withColumn("performance",
    when(col("marks") >= 85, "Excellent")
    .when(col("marks") >= 60, "Good")
    .when(col("marks") >= 40, "Average")
    .otherwise("At Risk")
)

df.show()

+----------+---------+-------+-----+--------+---------+-----------+
|student_id|     name|subject|marks|semester|exam_type|performance|
+----------+---------+-------+-----+--------+---------+-----------+
|         1|Keerthana|   Math|   85|    Sem1| Internal|  Excellent|
|         1|Keerthana|Physics|   78|    Sem1| External|       Good|
|         2|    Rahul|   Math|   45|    Sem1| Internal|    Average|
|         2|    Rahul|Physics|   55|    Sem1| External|    Average|
|         3|   Anjali|   Math|   92|    Sem2| Internal|  Excellent|
|         3|   Anjali|Physics|   88|    Sem2| External|  Excellent|
|         4|    Arjun|   Math|   30|    Sem2| Internal|    At Risk|
|         4|    Arjun|Physics|   40|    Sem2| External|    Average|
+----------+---------+-------+-----+--------+---------+-----------+



In [24]:
df.groupBy("performance").count().show()

+-----------+-----+
|performance|count|
+-----------+-----+
|  Excellent|    3|
|    Average|    3|
|       Good|    1|
|    At Risk|    1|
+-----------+-----+



In [25]:
from pyspark.sql.window import Window

window = Window.partitionBy("semester").orderBy(desc("marks"))

topper = df.withColumn("rank", row_number().over(window))
topper.filter("rank = 1").show()

+----------+---------+-------+-----+--------+---------+-----------+----+
|student_id|     name|subject|marks|semester|exam_type|performance|rank|
+----------+---------+-------+-----+--------+---------+-----------+----+
|         1|Keerthana|   Math|   85|    Sem1| Internal|  Excellent|   1|
|         3|   Anjali|   Math|   92|    Sem2| Internal|  Excellent|   1|
+----------+---------+-------+-----+--------+---------+-----------+----+



In [26]:
df.groupBy("semester", "subject") \
  .avg("marks") \
  .orderBy("semester") \
  .show()

+--------+-------+----------+
|semester|subject|avg(marks)|
+--------+-------+----------+
|    Sem1|Physics|      66.5|
|    Sem1|   Math|      65.0|
|    Sem2|   Math|      61.0|
|    Sem2|Physics|      64.0|
+--------+-------+----------+



In [27]:
df.filter(df.performance == "At Risk").show()

+----------+-----+-------+-----+--------+---------+-----------+
|student_id| name|subject|marks|semester|exam_type|performance|
+----------+-----+-------+-----+--------+---------+-----------+
|         4|Arjun|   Math|   30|    Sem2| Internal|    At Risk|
+----------+-----+-------+-----+--------+---------+-----------+

